# Q1 - Neural Network

In [84]:
#imports
import kagglehub
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import random

# Download Data
path = kagglehub.dataset_download('uciml/iris')
print ("path to dataset files:", path)
df = pd.read_csv(os.path.join(path, "Iris.csv"))
print (df.head())
print (df.shape)


path to dataset files: /Users/vip/.cache/kagglehub/datasets/uciml/iris/versions/2
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa
(150, 6)


In [85]:
#device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)


mps


In [86]:
# The MLP
class MLP(nn.Module):
    def __init__(self, input_size, num_neurons, output_size):
        super(MLP,self).__init__()
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, output_size)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


In [87]:
#Hyper Parameters
input_size = 4
output_size = 3
neurons = 16
learning_rate = 0.01



In [88]:
mp = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

# 80% of the data for training, 20% for testing.
l_train = int(len(df) * 0.8)
l_test = len(df) - l_train

indices = list(range(len(df)))
random.shuffle(indices)

train_indices = indices[:l_train]
test_indices = indices[l_train:]

X_train = [
    [df['PetalLengthCm'][i], df['PetalWidthCm'][i], df['SepalLengthCm'][i], df['SepalWidthCm'][i]]
    for i in train_indices
]

Y_train = [mp[df['Species'][i]] for i in train_indices]

X_test = [
    [df['PetalLengthCm'][i], df['PetalWidthCm'][i], df['SepalLengthCm'][i], df['SepalWidthCm'][i]]
    for i in test_indices
]
Y_test = [mp[df['Species'][i]] for i in test_indices]
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
Y_train = torch.tensor(Y_train).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
Y_test = torch.tensor(Y_test).to(device)


In [89]:
# initialize network
model = MLP(input_size,neurons,output_size).to(device)
print(model)

MLP(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=3, bias=True)
)


In [90]:
# Loss and Optimiser
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [91]:
# Training loop 
for i in range(1000):
    outputs = model(X_train)
    loss = criterion(outputs, Y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss.item():.4f}")

Iteration 0: Loss = 2.1341
Iteration 100: Loss = 0.2375
Iteration 200: Loss = 0.0854
Iteration 300: Loss = 0.0628
Iteration 400: Loss = 0.0550
Iteration 500: Loss = 0.0510
Iteration 600: Loss = 0.0485
Iteration 700: Loss = 0.0466
Iteration 800: Loss = 0.0453
Iteration 900: Loss = 0.0443


In [92]:
# Testing
with torch.no_grad():
    outputs = model(X_test)
    predictions = torch.argmax(outputs, dim=1)

correct = (predictions == Y_test).sum().item()
accuracy = correct / len(Y_test)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 96.67%


# Q2 Caeser Cipher

In [93]:
encrypted_text = "dyvkrwwrzbcksvz:xkdyeck@bv@rbvukr:ukw/btvckz:kbvruz:vccks/dykgzdy/edkr:ukgzdyz:kdyvkbvr,;mkdyv:kcyr,,kzdksvkdz;vkd/kcvdkdyvkczhkxv:d,v;v:kd/kg/b.nkdr.z:xk/buvbmke@/:kdyvkrtt/;@,zcyz:xk/wkdyvzbkuvczx:mkzk;riksvkceuuv:,ikdbr:c@/bdvuk/edk/wkdyzck@,rtvmkr:ukdyrdkr,,ki/ebkw/btvckz:kdyvkcr;vkdz;vksvk/:kdyvkwzv,ukd/k;vvdk;vl"

encrypted_postScript = "zkg/e,uksvkx,rukd/k.:/gkdyvk:r;vckr:ukaer,zdzvck/wkdyvkczhkxv:d,v;v:kgyztykrbvkd/krtt/;@,zcykdyvkuvczx:;v:dnkw/bkdyrdkzdk;riksvkzkcyr,,ksvkrs,vmke@/:k.:/g,vuxvk/wkdyvk@rbdzvcmkd/kxzfvki/ekc/;vkwebdyvbkrufztvk:vtvccrbikd/ksvkw/,,/gvukdyvbvz:"

characters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ', '.', ',', ';', ':', '/', '@']

def apply_shift(text, shift):
    characters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ', '.', ',', ';', ':', '/', '@']
    l = len(characters)
    lst = list(text)
    for i in range(len(lst)):
        char = lst[i]
        j = 0
        while(characters[j] != char):
            j += 1
        lst[i] = characters[(j+shift)%l]
    return ''.join(lst)

for i in range(1,len(characters)):
    print(i,":",apply_shift(encrypted_text, i))
    print("")



1 : ezwlsxxs cdltw /ylezfdlacwascwvls/vlx@cuwdl /lcwsv /wddlt@ezlh ez@fels/vlh ez /lezwlcws;:nlezw/ldzs;;l eltwle :wle@ldwelezwld ilyw/e;w:w/le@lh@c,oles, /yl@cvwcnlfa@/lezwlsuu@:a; dz /yl@xlezw clvwd y/nl l:sjltwldfvvw/;jlecs/da@cewvl@fel@xlez dla;suwnls/vlezsels;;lj@fclx@cuwdl /lezwlds:wle :wltwl@/lezwlx w;vle@l:wwel:wm

2 : f xmtyyt.demux.@zmf gembdxbtdxwmt@wmyadvxem.@mdxtw.@xeemuaf mi.f agfmt@wmi.f .@mf xmdxt:/omf x@me t::m.fmuxmf./xmfamexfmf xme.jmzx@f:x/x@mfamiad;pmft;.@zmadwxdomgba@mf xmtvva/b:.e .@zmaymf x.dmwxe.z@om.m/tkmuxmegwwx@:kmfdt@ebadfxwmagfmaymf .emb:tvxomt@wmf tfmt::mkagdmyadvxem.@mf xmet/xmf./xmuxma@mf xmy.x:wmfam/xxfm/xn

3 : g.ynuzzu,efnvy,a ng.hfnceycueyxnuaxnzbewyfn,aneyux,ayffnvbg.nj,g.bhgnuaxnj,g.,ang.yneyu/@png.yanf.u//n,gnvyng,@yngbnfygng.ynf,kn yag/y@yangbnjbe:qngu:,a nbexyepnhcbang.ynuwwb@c/,f.,a nbzng.y,enxyf, apn,n@ulnvynfhxxya/lngeuafcbegyxnbhgnbzng.,fnc/uwypnuaxng.ugnu//nlbhenzbewyfn,ang.ynfu@yng,@ynvynbang.ynz,y/xngbn@yygn@yo

4 : h,zov  v;fgowz;b.oh,i

In [94]:
# from the above it can be seen that a shift of 16 gives us readable english text
print(apply_shift(encrypted_text, 16))
print(apply_shift(encrypted_postScript,16))
# Thus the code can be broken by brute force.

the affairs being thus prepared and forces in readiness both without and within the realm, then shall it be time to set the six gentlemen to work; taking order, upon the accomplishing of their design, i may be suddenly transported out of this place, and that all your forces in the same time be on the field to meet me.
i would be glad to know the names and qualities of the six gentlemen which are to accomplish the designment; for that it may be i shall be able, upon knowledge of the parties, to give you some further advice necessary to be followed therein


### Now using frequency analysis.

In [95]:
letterFrequency = {'E' : 12.0,
'T' : 9.10,
'A' : 8.12,
'O' : 7.68,
'I' : 7.31,
'N' : 6.95,
'S' : 6.28,
'R' : 6.02,
'H' : 5.92,
'D' : 4.32,
'L' : 3.98,
'U' : 2.88,
'C' : 2.71,
'M' : 2.61,
'F' : 2.30,
'Y' : 2.11,
'W' : 2.09,
'G' : 2.03,
'P' : 1.82,
'B' : 1.49,
'V' : 1.11,
'K' : 0.69,
'X' : 0.17,
'Q' : 0.11,
'J' : 0.10,
'Z' : 0.07 }

In [96]:
# Frequencies in the code
code_frequency = {}
for char in encrypted_text:
    if (char not in code_frequency.keys()):
        code_frequency[char] = 0
    code_frequency[char] += 1

print(sorted(code_frequency.items(), key = lambda x : x[1], reverse=True))

[('k', 59), ('v', 35), ('d', 29), ('z', 20), (':', 18), ('r', 17), ('/', 17), ('y', 16), ('c', 15), ('b', 14), ('u', 11), (',', 10), (';', 9), ('w', 7), ('e', 6), ('@', 6), ('s', 5), ('x', 5), ('t', 5), ('m', 4), ('g', 3), ('i', 3), ('.', 2), ('h', 1), ('n', 1), ('l', 1)]


In [97]:
print(len(encrypted_text)) # 319
# Average word in english language is around 5 letters long, so expect around 319/(5+1) spaces or full stops ~ 53, so k most likely codes for space
# Mapping 
mapping = {
    'k' : ' ',
    'v' : 'e',
    'd' : 't'
}
def apply_mapping(text, mapping):
    lst = list(text)
    for i in range(len(lst)):
        if lst[i] in mapping.keys():
            lst[i] = mapping[lst[i]]
    return ''.join(lst)

print(apply_mapping(encrypted_text,mapping))

319
tye rwwrzbc sez:x tyec @be@rbeu r:u w/btec z: beruz:ecc s/ty gzty/et r:u gztyz: tye ber,;m tye: cyr,, zt se tz;e t/ cet tye czh xe:t,e;e: t/ g/b.n tr.z:x /buebm e@/: tye rtt/;@,zcyz:x /w tyezb ueczx:m z ;ri se ceuue:,i tbr:c@/bteu /et /w tyzc @,rtem r:u tyrt r,, i/eb w/btec z: tye cr;e tz;e se /: tye wze,u t/ ;eet ;el


In [ ]:
# since first word is 'tye', we can guess y codes for h, to make the word 'the'.
mapping['y'] = 'h'
print(mapping)
# Since we know its a Caesar cipher and we can see that all the characters we guessed are shifted by 16, we can conclude that this is how they were encoded.
for i in range(len(characters)):
    mapping[characters[i]] = characters[(i+16)%len(characters)]
print(mapping)
print(apply_mapping(encrypted_text, mapping))
print(apply_mapping(encrypted_postScript, mapping))

{'k': ' ', 'v': 'e', 'd': 't', 'y': 'h'}
{'k': ' ', 'v': 'e', 'd': 't', 'y': 'h', 'a': 'q', 'b': 'r', 'c': 's', 'e': 'u', 'f': 'v', 'g': 'w', 'h': 'x', 'i': 'y', 'j': 'z', 'l': '.', 'm': ',', 'n': ';', 'o': ':', 'p': '/', 'q': '@', 'r': 'a', 's': 'b', 't': 'c', 'u': 'd', 'w': 'f', 'x': 'g', 'z': 'i', ' ': 'j', '.': 'k', ',': 'l', ';': 'm', ':': 'n', '/': 'o', '@': 'p'}
the affairs being thus prepared and forces in readiness both without and within the realm, then shall it be time to set the six gentlemen to work; taking order, upon the accomplishing of their design, i may be suddenly transported out of this place, and that all your forces in the same time be on the field to meet me.
i would be glad to know the names and qualities of the six gentlemen which are to accomplish the designment; for that it may be i shall be able, upon knowledge of the parties, to give you some further advice necessary to be followed therein
